In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, month, year

In [3]:
spark = SparkSession.builder \
    .appName("Retail Sales Week 3") \
    .getOrCreate()

In [4]:
from google.colab import files
uploaded = files.upload()

Saving large_sales.csv to large_sales.csv


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Retail Sales").getOrCreate()

df = spark.read.csv("large_sales.csv", header=True, inferSchema=True)
df.show(5)

+-------+--------+---------------+----------+------------+--------+-----+-----+-------+----------+
|sale_id|store_id|     store_name|product_id|product_name|quantity|price| cost|returns| sale_date|
+-------+--------+---------------+----------+------------+--------+-----+-----+-------+----------+
|      1|     104|   Mumbai Store|       205|       Mouse|       7|54843|42755|      2|2026-05-02|
|      2|     103|Hyderabad Store|       207|      Tablet|       3|22462|19314|      3|2026-04-14|
|      3|     103|Hyderabad Store|       206|     Monitor|       5| 1269| 1036|      1|2026-02-07|
|      4|     102|Bangalore Store|       208|     Printer|       4|39688|31164|      1|2026-01-22|
|      5|     105|    Delhi Store|       204|    Keyboard|       1|41934|32857|      1|2026-01-15|
+-------+--------+---------------+----------+------------+--------+-----+-----+-------+----------+
only showing top 5 rows


In [6]:
df = df.withColumn("revenue", col("quantity") * col("price"))
df = df.withColumn("profit", col("revenue") - (col("quantity") * col("cost")))
df = df.withColumn("sale_month", month(col("sale_date")))
df = df.withColumn("sale_year", year(col("sale_date")))

df.show(5)

+-------+--------+---------------+----------+------------+--------+-----+-----+-------+----------+-------+------+----------+---------+
|sale_id|store_id|     store_name|product_id|product_name|quantity|price| cost|returns| sale_date|revenue|profit|sale_month|sale_year|
+-------+--------+---------------+----------+------------+--------+-----+-----+-------+----------+-------+------+----------+---------+
|      1|     104|   Mumbai Store|       205|       Mouse|       7|54843|42755|      2|2026-05-02| 383901| 84616|         5|     2026|
|      2|     103|Hyderabad Store|       207|      Tablet|       3|22462|19314|      3|2026-04-14|  67386|  9444|         4|     2026|
|      3|     103|Hyderabad Store|       206|     Monitor|       5| 1269| 1036|      1|2026-02-07|   6345|  1165|         2|     2026|
|      4|     102|Bangalore Store|       208|     Printer|       4|39688|31164|      1|2026-01-22| 158752| 34096|         1|     2026|
|      5|     105|    Delhi Store|       204|    Keyboa

In [7]:
product_summary = df.groupBy(
    "product_id",
    "product_name"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("returns").alias("total_returns"),
    sum("revenue").alias("total_revenue"),
    avg("profit").alias("avg_profit")
)

product_summary.show(truncate=False)

+----------+------------+-------------------+-------------+-------------+------------------+
|product_id|product_name|total_quantity_sold|total_returns|total_revenue|avg_profit        |
+----------+------------+-------------------+-------------+-------------+------------------+
|203       |Headphones  |2496               |926          |78451352     |30405.108108108107|
|201       |Laptop      |2479               |985          |74833697     |30944.408866995072|
|206       |Monitor     |2329               |909          |69547883     |28467.110738255033|
|204       |Keyboard    |2623               |966          |79812843     |31600.11370716511 |
|202       |Mobile      |2557               |950          |76548490     |30196.198127925116|
|207       |Tablet      |2577               |993          |77748998     |30764.426771653543|
|208       |Printer     |2537               |939          |76509322     |30669.83922829582 |
|205       |Mouse       |2443               |927          |73536606   

In [8]:
underperforming_products = product_summary.filter(
    (col("total_revenue") < 500000) | (col("total_returns") > 200)
)

underperforming_products.show(truncate=False)

+----------+------------+-------------------+-------------+-------------+------------------+
|product_id|product_name|total_quantity_sold|total_returns|total_revenue|avg_profit        |
+----------+------------+-------------------+-------------+-------------+------------------+
|203       |Headphones  |2496               |926          |78451352     |30405.108108108107|
|201       |Laptop      |2479               |985          |74833697     |30944.408866995072|
|206       |Monitor     |2329               |909          |69547883     |28467.110738255033|
|204       |Keyboard    |2623               |966          |79812843     |31600.11370716511 |
|202       |Mobile      |2557               |950          |76548490     |30196.198127925116|
|207       |Tablet      |2577               |993          |77748998     |30764.426771653543|
|208       |Printer     |2537               |939          |76509322     |30669.83922829582 |
|205       |Mouse       |2443               |927          |73536606   

In [9]:
store_monthly_revenue = df.groupBy(
    "store_id",
    "store_name",
    "sale_year",
    "sale_month"
).agg(
    sum("revenue").alias("monthly_revenue")
)

store_monthly_revenue.show(truncate=False)

+--------+---------------+---------+----------+---------------+
|store_id|store_name     |sale_year|sale_month|monthly_revenue|
+--------+---------------+---------+----------+---------------+
|104     |Mumbai Store   |2026     |2         |14056116       |
|103     |Hyderabad Store|2026     |5         |20044036       |
|104     |Mumbai Store   |2026     |5         |20443913       |
|103     |Hyderabad Store|2026     |1         |16845463       |
|104     |Mumbai Store   |2026     |1         |21779714       |
|102     |Bangalore Store|2026     |2         |18586733       |
|102     |Bangalore Store|2026     |5         |20270290       |
|105     |Delhi Store    |2026     |5         |19842223       |
|104     |Mumbai Store   |2026     |6         |18901958       |
|105     |Delhi Store    |2026     |2         |18125923       |
|104     |Mumbai Store   |2026     |4         |22105565       |
|101     |Chennai Store  |2026     |4         |23989437       |
|101     |Chennai Store  |2026     |1   

In [10]:
avg_monthly_revenue_by_store = store_monthly_revenue.groupBy(
    "store_id",
    "store_name"
).agg(
    avg("monthly_revenue").alias("avg_monthly_revenue")
)

avg_monthly_revenue_by_store.show(truncate=False)

+--------+---------------+--------------------+
|store_id|store_name     |avg_monthly_revenue |
+--------+---------------+--------------------+
|105     |Delhi Store    |2.0574724166666668E7|
|102     |Bangalore Store|2.0310360333333332E7|
|104     |Mumbai Store   |2.00293565E7        |
|101     |Chennai Store  |2.0546062333333332E7|
|103     |Hyderabad Store|1.9704361833333332E7|
+--------+---------------+--------------------+



In [11]:
underperforming_products.toPandas().to_csv(
    "underperforming_products_output.csv",
    index=False
)

avg_monthly_revenue_by_store.toPandas().to_csv(
    "store_avg_monthly_revenue.csv",
    index=False
)